# Análise Exploratória — Acidentes PRF 2024/2025/2026

Este notebook documenta o pipeline de Ciência de Dados utilizado no dashboard interativo em Dash.  
A análise integra os arquivos brutos de **2024, 2025 e 2026**, permitindo comparar os três períodos e identificar padrões de volume, gravidade, localização, horário, clima, causas e rodovias.

> **Observação importante:** a base de 2026 está parcial no arquivo recebido, com registros até **30/04/2026**. Por isso, as comparações anuais devem considerar que 2024 e 2025 são anos completos, enquanto 2026 representa apenas parte do ano.


## 1. Aquisição de dados

Nesta etapa, os arquivos brutos disponibilizados pela PRF são carregados diretamente no notebook.  
Foram utilizados três arquivos distintos:

- `datatran2024.csv`
- `datatran2025.csv`
- `datatran2026.csv`

A leitura é feita com `pandas`, utilizando separador `;` e tratamento de codificação.  
O código abaixo também foi preparado para localizar os arquivos em diferentes estruturas de pasta, como a raiz do projeto, `data/bruto/` ou a pasta anterior ao notebook.


In [1]:

# Instala automaticamente as bibliotecas caso o ambiente do Jupyter ainda não tenha os pacotes.
# Isso evita o erro: ModuleNotFoundError: No module named 'pandas'

import sys
import subprocess
import importlib.util

pacotes = ["pandas", "plotly", "numpy", "nbformat"]

for pacote in pacotes:
    if importlib.util.find_spec(pacote) is None:
        print(f"Instalando {pacote}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pacote])

from pathlib import Path
import warnings
import re

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Candidatos de diretório para facilitar a execução tanto na raiz do projeto quanto dentro da pasta notebooks/
CANDIDATE_DIRS = [
    Path.cwd(),
    Path.cwd() / "data" / "bruto",
    Path.cwd().parent,
    Path.cwd().parent / "data" / "bruto",
    Path("/mnt/data")
]

def encontrar_arquivo(nome_arquivo):
    for pasta in CANDIDATE_DIRS:
        caminho = pasta / nome_arquivo
        if caminho.exists():
            return caminho
    raise FileNotFoundError(
        f"Arquivo {nome_arquivo} não encontrado. "
        "Coloque os CSVs na raiz do projeto ou em data/bruto/."
    )

arquivos = {
    2024: encontrar_arquivo("datatran2024.csv"),
    2025: encontrar_arquivo("datatran2025.csv"),
    2026: encontrar_arquivo("datatran2026.csv")
}

arquivos


{2024: WindowsPath('d:/estudos/Faculdade/PROJETO FINAL BANCO/ultima versão/projeto_prf_dashboard_2024_2026_completo/data/bruto/datatran2024.csv'),
 2025: WindowsPath('d:/estudos/Faculdade/PROJETO FINAL BANCO/ultima versão/projeto_prf_dashboard_2024_2026_completo/data/bruto/datatran2025.csv'),
 2026: WindowsPath('d:/estudos/Faculdade/PROJETO FINAL BANCO/ultima versão/projeto_prf_dashboard_2024_2026_completo/data/bruto/datatran2026.csv')}

In [2]:

def ler_csv_prf(caminho, ano):
    """
    Lê um arquivo bruto da PRF e adiciona a coluna de ano de origem.
    A função tenta latin1 e utf-8-sig para evitar erro de codificação.
    """
    try:
        base = pd.read_csv(caminho, sep=";", encoding="latin1")
    except UnicodeDecodeError:
        base = pd.read_csv(caminho, sep=";", encoding="utf-8-sig")

    base["ano_origem"] = ano
    return base

bases_brutas = []

for ano, caminho in arquivos.items():
    base_ano = ler_csv_prf(caminho, ano)
    bases_brutas.append(base_ano)
    print(f"{ano}: {len(base_ano):,} registros | {base_ano.shape[1]} colunas | arquivo: {caminho.name}".replace(",", "."))

display(bases_brutas[0].head())


2024: 73.156 registros | 31 colunas | arquivo: datatran2024.csv
2025: 72.529 registros | 31 colunas | arquivo: datatran2025.csv
2026: 23.475 registros | 31 colunas | arquivo: datatran2026.csv


,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop,ano_origem
0,"571,789.00",2024-01-01,segunda-feira,03:56:00,ES,101,38,CONCEICAO DA BARRA,Ultrapassagem Indevida,Colisão lateral sentido oposto,NaN,Plena Noite,Crescente,Céu Claro,Simples,Reta,Não,3,0,0,1,1,1,1,3,-18.48,-39.92,SPRF-ES,DEL04-ES,UOP02-DEL04-ES,2024
1,"571,804.00",2024-01-01,segunda-feira,04:50:00,PI,343,185,PIRIPIRI,Manobra de mudança de faixa,Colisão frontal,Com Vítimas Fatais,Amanhecer,Decrescente,Céu Claro,Simples,Reta,Sim,2,1,0,0,1,0,0,2,-4.30,-41.77,SPRF-PI,DEL02-PI,UOP01-DEL02-PI,2024
2,"571,806.00",2024-01-01,segunda-feira,04:30:00,BA,116,578,BREJOES,Ingestão de álcool pelo condutor,Colisão frontal,Com Vítimas Fatais,Plena Noite,Decrescente,Céu Claro,Simples,Curva,Não,3,1,0,0,1,2,0,4,-13.07,-39.96,SPRF-BA,DEL03-BA,UOP02-DEL03-BA,2024
3,"571,818.00",2024-01-01,segunda-feira,06:30:00,SE,101,18,MALHADA DOS BOIS,Reação tardia ou ineficiente do condutor,Saída de leito carroçável,Com Vítimas Feridas,Amanhecer,Crescente,Céu Claro,Dupla,Declive;Reta,Não,2,0,0,1,0,2,1,3,-10.36,-36.91,SPRF-SE,DEL02-SE,UOP02-DEL02-SE,2024
4,"571,838.00",2024-01-01,segunda-feira,05:00:00,MT,364,240,RONDONOPOLIS,Condutor deixou de manter distância do veículo...,Colisão traseira,Sem Vítimas,Pleno dia,Crescente,Céu Claro,Dupla,Reta,Não,3,0,0,0,2,1,0,3,-16.18,-54.79,SPRF-MT,DEL02-MT,UOP01-DEL02-MT,2024


## 2. Integração de dados

Como os três arquivos possuem a mesma estrutura de colunas, a integração principal é feita por **concatenação vertical**.  
Essa etapa cria uma base única de análise, mantendo a identificação do ano de origem para permitir comparação entre 2024, 2025 e 2026.

Também é realizada uma validação das colunas para verificar se os arquivos possuem estrutura compatível.


In [3]:

# Verificação da estrutura dos arquivos
colunas_por_ano = {ano: set(base.columns) for ano, base in zip(arquivos.keys(), bases_brutas)}
colunas_referencia = set(bases_brutas[0].columns)

for ano, colunas in colunas_por_ano.items():
    diferencas = colunas.symmetric_difference(colunas_referencia)
    print(f"Ano {ano}: {len(colunas)} colunas | Diferenças em relação à referência: {len(diferencas)}")

# Integração por concatenação
df_raw = pd.concat(bases_brutas, ignore_index=True)

print("Quantidade total de registros brutos:", f"{len(df_raw):,}".replace(",", "."))
print("Quantidade total de colunas:", df_raw.shape[1])

df_raw.groupby("ano_origem").size().reset_index(name="registros")


Ano 2024: 31 colunas | Diferenças em relação à referência: 0
Ano 2025: 31 colunas | Diferenças em relação à referência: 0
Ano 2026: 31 colunas | Diferenças em relação à referência: 0
Quantidade total de registros brutos: 169.160
Quantidade total de colunas: 31


,ano_origem,registros
0,2024,73156
1,2025,72529
2,2026,23475


### Interpretação

A integração dos três arquivos amplia a análise temporal e permite comparar anos completos com o período parcial de 2026.  
A utilização de três bases também fortalece o projeto, pois atende melhor ao requisito de trabalhar com arquivos distintos e possibilita avaliar tendências, mudanças e padrões ao longo do tempo.


## 3. Limpeza e tratamento dos dados

Nesta etapa, os dados brutos são preparados para análise.  
Os principais tratamentos realizados foram:

- padronização dos nomes das colunas;
- conversão da data para formato `datetime`;
- conversão do horário para hora numérica;
- tratamento de latitude, longitude e quilometragem com vírgula decimal;
- conversão de variáveis numéricas;
- preenchimento de valores ausentes em campos categóricos;
- remoção de duplicidades pelo identificador `id`;
- correção de possíveis valores numéricos inválidos.

Essa etapa é essencial para evitar distorções nos filtros, gráficos e indicadores do dashboard.


In [4]:

# Diagnóstico inicial de valores ausentes nas principais colunas
colunas_analiticas = [
    "id", "data_inversa", "dia_semana", "horario", "uf", "br", "km",
    "municipio", "causa_acidente", "tipo_acidente", "classificacao_acidente",
    "fase_dia", "condicao_metereologica", "tipo_pista",
    "pessoas", "mortos", "feridos_leves", "feridos_graves", "feridos",
    "veiculos", "latitude", "longitude"
]

colunas_existentes = [col for col in colunas_analiticas if col in df_raw.columns]

diagnostico_nulos = (
    df_raw[colunas_existentes]
    .isna()
    .sum()
    .reset_index()
)

diagnostico_nulos.columns = ["coluna", "valores_ausentes"]
diagnostico_nulos["percentual_%"] = (diagnostico_nulos["valores_ausentes"] / len(df_raw) * 100).round(2)

diagnostico_nulos.sort_values("valores_ausentes", ascending=False).head(15)


,coluna,valores_ausentes,percentual_%
10,classificacao_acidente,3,0.00
0,id,0,0.00
12,condicao_metereologica,0,0.00
20,latitude,0,0.00
19,veiculos,0,0.00
18,feridos,0,0.00
17,feridos_graves,0,0.00
16,feridos_leves,0,0.00
15,mortos,0,0.00
14,pessoas,0,0.00


In [5]:

def normalizar_numero(valor):
    """
    Converte campos numéricos da PRF que podem vir com vírgula decimal.
    Exemplo: '-23,48586772' -> -23.48586772
    """
    if pd.isna(valor):
        return np.nan

    texto = str(valor).strip().replace(",", ".")
    texto = re.sub(r"[^0-9\.\-]", "", texto)

    try:
        return float(texto)
    except ValueError:
        return np.nan


MAPA_REGIOES = {
    "AC": "Norte", "AP": "Norte", "AM": "Norte", "PA": "Norte", "RO": "Norte", "RR": "Norte", "TO": "Norte",
    "AL": "Nordeste", "BA": "Nordeste", "CE": "Nordeste", "MA": "Nordeste", "PB": "Nordeste",
    "PE": "Nordeste", "PI": "Nordeste", "RN": "Nordeste", "SE": "Nordeste",
    "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MT": "Centro-Oeste", "MS": "Centro-Oeste",
    "ES": "Sudeste", "MG": "Sudeste", "RJ": "Sudeste", "SP": "Sudeste",
    "PR": "Sul", "RS": "Sul", "SC": "Sul"
}

MAPA_MESES = {
    1: "Jan", 2: "Fev", 3: "Mar", 4: "Abr", 5: "Mai", 6: "Jun",
    7: "Jul", 8: "Ago", 9: "Set", 10: "Out", 11: "Nov", 12: "Dez"
}

def classificar_periodo_hora(hora):
    if pd.isna(hora):
        return "Não informado"

    hora = int(hora)

    if 0 <= hora <= 5:
        return "Madrugada"
    elif 6 <= hora <= 11:
        return "Manhã"
    elif 12 <= hora <= 17:
        return "Tarde"
    elif 18 <= hora <= 23:
        return "Noite"
    else:
        return "Não informado"


def classificar_gravidade(linha):
    """
    Cria uma classificação própria de gravidade baseada nos campos de vítimas.
    Isso evita depender apenas da coluna classificacao_acidente, que pode ter nulos.
    """
    if linha["mortos"] > 0:
        return "Fatal"
    elif linha["feridos_graves"] > 0:
        return "Com feridos graves"
    elif linha["feridos_leves"] > 0 or linha["feridos"] > 0:
        return "Com feridos leves"
    else:
        return "Sem vítimas"


def preparar_dados_prf(df_entrada):
    df = df_entrada.copy()

    # Padronização de nomes de colunas
    df.columns = [col.strip().lower() for col in df.columns]

    # Ano de referência
    if "ano" not in df.columns:
        df["ano"] = df["ano_origem"]

    # Datas e horários
    df["data_inversa"] = pd.to_datetime(df["data_inversa"], errors="coerce")
    df["ano"] = df["ano"].fillna(df["data_inversa"].dt.year).astype(int)
    df["hora"] = pd.to_datetime(df["horario"], format="%H:%M:%S", errors="coerce").dt.hour

    # Conversão de campos numéricos principais
    colunas_numericas = [
        "id", "br", "pessoas", "mortos", "feridos_leves", "feridos_graves",
        "ilesos", "ignorados", "feridos", "veiculos"
    ]

    for col in colunas_numericas:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Latitude, longitude e km
    for col in ["latitude", "longitude", "km"]:
        if col in df.columns:
            df[col] = df[col].apply(normalizar_numero)

    # Tratamento de textos
    colunas_texto = [
        "uf", "municipio", "causa_acidente", "tipo_acidente", "classificacao_acidente",
        "fase_dia", "sentido_via", "condicao_metereologica", "tipo_pista",
        "tracado_via", "uso_solo", "regional", "delegacia", "uop"
    ]

    for col in colunas_texto:
        if col in df.columns:
            df[col] = (
                df[col]
                .fillna("Não informado")
                .astype(str)
                .str.strip()
            )

    # Dia da semana padronizado
    df["dia_semana"] = (
        df["dia_semana"]
        .fillna("Não informado")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    # Remoção de duplicidades por id
    antes = len(df)
    df = df.drop_duplicates(subset=["id"])
    depois = len(df)

    print(f"Duplicidades removidas por id: {antes - depois}")

    # Evita valores negativos em contagens
    for col in ["pessoas", "mortos", "feridos_leves", "feridos_graves", "ilesos", "ignorados", "feridos", "veiculos"]:
        if col in df.columns:
            df.loc[df[col] < 0, col] = np.nan
            df[col] = df[col].fillna(0).astype(int)

    return df

df = preparar_dados_prf(df_raw)

print("Registros após tratamento:", f"{len(df):,}".replace(",", "."))
display(df.head())


Duplicidades removidas por id: 0
Registros após tratamento: 169.160


,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop,ano_origem,ano,hora
0,"571,789.00",2024-01-01,segunda-feira,03:56:00,ES,101,38.00,CONCEICAO DA BARRA,Ultrapassagem Indevida,Colisão lateral sentido oposto,Não informado,Plena Noite,Crescente,Céu Claro,Simples,Reta,Não,3,0,0,1,1,1,1,3,-18.48,-39.92,SPRF-ES,DEL04-ES,UOP02-DEL04-ES,2024,2024,3
1,"571,804.00",2024-01-01,segunda-feira,04:50:00,PI,343,185.00,PIRIPIRI,Manobra de mudança de faixa,Colisão frontal,Com Vítimas Fatais,Amanhecer,Decrescente,Céu Claro,Simples,Reta,Sim,2,1,0,0,1,0,0,2,-4.30,-41.77,SPRF-PI,DEL02-PI,UOP01-DEL02-PI,2024,2024,4
2,"571,806.00",2024-01-01,segunda-feira,04:30:00,BA,116,578.00,BREJOES,Ingestão de álcool pelo condutor,Colisão frontal,Com Vítimas Fatais,Plena Noite,Decrescente,Céu Claro,Simples,Curva,Não,3,1,0,0,1,2,0,4,-13.07,-39.96,SPRF-BA,DEL03-BA,UOP02-DEL03-BA,2024,2024,4
3,"571,818.00",2024-01-01,segunda-feira,06:30:00,SE,101,18.00,MALHADA DOS BOIS,Reação tardia ou ineficiente do condutor,Saída de leito carroçável,Com Vítimas Feridas,Amanhecer,Crescente,Céu Claro,Dupla,Declive;Reta,Não,2,0,0,1,0,2,1,3,-10.36,-36.91,SPRF-SE,DEL02-SE,UOP02-DEL02-SE,2024,2024,6
4,"571,838.00",2024-01-01,segunda-feira,05:00:00,MT,364,240.00,RONDONOPOLIS,Condutor deixou de manter distância do veículo...,Colisão traseira,Sem Vítimas,Pleno dia,Crescente,Céu Claro,Dupla,Reta,Não,3,0,0,0,2,1,0,3,-16.18,-54.79,SPRF-MT,DEL02-MT,UOP01-DEL02-MT,2024,2024,5


### Interpretação

A limpeza dos dados reduz problemas que poderiam prejudicar a análise, como categorias vazias, campos numéricos lidos como texto, duplicidades e datas em formato inadequado.  
Com os dados tratados, os indicadores passam a representar ocorrências únicas e comparáveis entre os anos.

A padronização é especialmente importante porque o dashboard utiliza filtros por ano, UF, condição meteorológica, fase do dia e tipo de acidente. Sem essa preparação, o usuário poderia visualizar categorias duplicadas ou indicadores inconsistentes.


## 4. Transformação de dados

Após a limpeza, foram criadas variáveis derivadas para enriquecer a análise.  
Essas variáveis permitem investigar padrões que não aparecem diretamente nas colunas originais.

Foram criadas, entre outras:

- mês e nome do mês;
- período do dia a partir da hora;
- indicador de fim de semana;
- indicador de acidente fatal;
- total de vítimas;
- taxa de gravidade;
- classificação analítica de gravidade;
- região do Brasil a partir da UF.


In [6]:

# Criação de variáveis temporais
df["mes"] = df["data_inversa"].dt.month.astype("Int64")
df["mes_nome"] = df["mes"].map(MAPA_MESES)
df["periodo_hora"] = df["hora"].apply(classificar_periodo_hora)

# Variáveis de calendário
df["fim_de_semana"] = df["dia_semana"].isin(["sábado", "sabado", "domingo"])

# Variáveis de gravidade
df["acidente_fatal"] = df["mortos"] > 0
df["total_vitimas"] = df[["mortos", "feridos_leves", "feridos_graves"]].sum(axis=1)

# Índice simples ponderado de gravidade:
# morto = peso 3, ferido grave = peso 2, ferido leve = peso 1
df["taxa_gravidade"] = (
    df["mortos"] * 3 +
    df["feridos_graves"] * 2 +
    df["feridos_leves"]
)

df["gravidade"] = df.apply(classificar_gravidade, axis=1)

# Região
df["regiao"] = df["uf"].map(MAPA_REGIOES).fillna("Não informado")

# Salva a base tratada em uma pasta compatível com o projeto
saida_tratada = Path.cwd() / "data" / "tratado" / "acidentes_prf_2024_2025_2026_tratado.csv"

try:
    saida_tratada.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(saida_tratada, sep=";", index=False, encoding="utf-8-sig")
    print(f"Base tratada salva em: {saida_tratada}")
except Exception as erro:
    print("Não foi possível salvar automaticamente a base tratada neste diretório.")
    print("Erro:", erro)

variaveis_criadas = [
    "ano", "mes", "mes_nome", "hora", "periodo_hora", "fim_de_semana",
    "acidente_fatal", "total_vitimas", "taxa_gravidade", "gravidade", "regiao"
]

df[variaveis_criadas].head()


Base tratada salva em: d:\estudos\Faculdade\PROJETO FINAL BANCO\ultima versão\projeto_prf_dashboard_2024_2026_completo\notebooks\data\tratado\acidentes_prf_2024_2025_2026_tratado.csv


,ano,mes,mes_nome,hora,periodo_hora,fim_de_semana,acidente_fatal,total_vitimas,taxa_gravidade,gravidade,regiao
0,2024,1,Jan,3,Madrugada,False,False,1,2,Com feridos graves,Sudeste
1,2024,1,Jan,4,Madrugada,False,True,1,3,Fatal,Nordeste
2,2024,1,Jan,4,Madrugada,False,True,1,3,Fatal,Nordeste
3,2024,1,Jan,6,Manhã,False,False,1,2,Com feridos graves,Nordeste
4,2024,1,Jan,5,Madrugada,False,False,0,0,Sem vítimas,Centro-Oeste


### Interpretação

A transformação cria uma base mais analítica e adequada ao dashboard.  
Por exemplo, a variável `acidente_fatal` permite medir a proporção de acidentes com morte, enquanto `periodo_hora` e `fim_de_semana` permitem investigar se determinados períodos concentram maior severidade.

A variável `taxa_gravidade` não substitui análises oficiais, mas funciona como um índice exploratório para comparar ocorrências considerando pesos diferentes para mortos, feridos graves e feridos leves.


## 5. Estatísticas descritivas gerais

Antes das análises gráficas, é importante observar os indicadores consolidados do período.  
Essa visão mostra o tamanho da base, o recorte temporal e a dimensão geral dos acidentes.


In [7]:

resumo = pd.DataFrame({
    "Indicador": [
        "Total de acidentes",
        "Período analisado",
        "Acidentes em 2024",
        "Acidentes em 2025",
        "Acidentes em 2026",
        "Total de mortos",
        "Total de feridos",
        "Acidentes fatais",
        "Taxa geral de letalidade (%)",
        "Média de feridos por acidente"
    ],
    "Valor": [
        f"{len(df):,}".replace(",", "."),
        f"{df['data_inversa'].min().date()} a {df['data_inversa'].max().date()}",
        f"{(df['ano'] == 2024).sum():,}".replace(",", "."),
        f"{(df['ano'] == 2025).sum():,}".replace(",", "."),
        f"{(df['ano'] == 2026).sum():,}".replace(",", "."),
        f"{df['mortos'].sum():,}".replace(",", "."),
        f"{df['feridos'].sum():,}".replace(",", "."),
        f"{df['acidente_fatal'].sum():,}".replace(",", "."),
        round(df["mortos"].sum() / len(df) * 100, 2),
        round(df["feridos"].sum() / len(df), 2)
    ]
})

resumo


,Indicador,Valor
0,Total de acidentes,169.160
1,Período analisado,2024-01-01 a 2026-04-30
2,Acidentes em 2024,73.156
3,Acidentes em 2025,72.529
4,Acidentes em 2026,23.475
5,Total de mortos,14.121
6,Total de feridos,195.633
7,Acidentes fatais,12.045
8,Taxa geral de letalidade (%),8.35
9,Média de feridos por acidente,1.16


### Insight 1 — Gravidade geral do problema

No período analisado, foram registrados **169.160 acidentes**, com **14.121 mortos** e **195.633 feridos**.  
A taxa geral de letalidade foi de aproximadamente **8,35%**, indicando que a análise não deve se limitar ao volume de acidentes.

A inclusão de 2024 ampliou a base e fortaleceu a comparação temporal, pois agora é possível observar dois anos completos (**2024 e 2025**) e um ano parcial (**2026 até abril**). Essa distinção é relevante para evitar interpretações equivocadas sobre queda ou aumento anual.


## 6. Tendência temporal dos acidentes

A análise mensal permite identificar variações ao longo do tempo, efeitos de sazonalidade, férias, feriados e períodos de maior circulação nas rodovias.  
Como 2026 é parcial, o gráfico deve ser interpretado com cuidado: ele mostra os meses disponíveis, mas não representa o ano completo.


In [8]:

MES_ORDEM = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]

mensal = (
    df.groupby(["ano", "mes", "mes_nome"], as_index=False)
    .agg(
        acidentes=("id", "count"),
        mortos=("mortos", "sum"),
        feridos=("feridos", "sum")
    )
    .sort_values(["ano", "mes"])
)

mensal["mes_nome"] = pd.Categorical(mensal["mes_nome"], categories=MES_ORDEM, ordered=True)

fig = px.line(
    mensal,
    x="mes_nome",
    y="acidentes",
    color="ano",
    markers=True,
    title="Evolução mensal dos acidentes por ano",
    labels={"mes_nome": "Mês", "acidentes": "Acidentes", "ano": "Ano"}
)

fig.update_layout(template="plotly_white")
fig.show()

mensal.sort_values("acidentes", ascending=False).head(12)


,ano,mes,mes_nome,acidentes,mortos,feridos
23,2025,12,Dez,6788,572,8100
11,2024,12,Dez,6587,632,7861
9,2024,10,Out,6406,493,7521
6,2024,7,Jul,6401,537,7383
21,2025,10,Out,6252,494,7231
19,2025,8,Ago,6246,554,7006
18,2025,7,Jul,6238,536,7129
5,2024,6,Jun,6225,535,7030
8,2024,9,Set,6222,532,6979
22,2025,11,Nov,6209,498,7103


In [9]:

# Comparação anual consolidada
comparativo_ano = (
    df.groupby("ano", as_index=False)
    .agg(
        acidentes=("id", "count"),
        mortos=("mortos", "sum"),
        feridos=("feridos", "sum"),
        acidentes_fatais=("acidente_fatal", "sum")
    )
)

comparativo_ano["letalidade_%"] = (comparativo_ano["mortos"] / comparativo_ano["acidentes"] * 100).round(2)

fig = px.bar(
    comparativo_ano,
    x="ano",
    y="acidentes",
    text="acidentes",
    title="Comparação anual de acidentes — atenção: 2026 parcial",
    labels={"ano": "Ano", "acidentes": "Acidentes"}
)

fig.update_layout(template="plotly_white")
fig.show()

comparativo_ano


,ano,acidentes,mortos,feridos,acidentes_fatais,letalidade_%
0,2024,73156,6160,84526,5222,8.42
1,2025,72529,6043,83550,5210,8.33
2,2026,23475,1918,27557,1613,8.17


### Insight 2 — Tendência temporal

O mês com maior volume de acidentes na base integrada foi **Dez/2025**, com **6.788 ocorrências**.  
Esse tipo de concentração pode estar associado a períodos de maior fluxo rodoviário, férias, datas comemorativas e deslocamentos de fim de ano.

A comparação anual mostra que **2024 e 2025 possuem bases completas**, enquanto **2026 está parcial até abril**. Portanto, o menor volume de 2026 não deve ser interpretado automaticamente como redução anual dos acidentes, mas sim como consequência do recorte disponível.


### 6.1 Comparação justa: Jan–Abr entre 2024, 2025 e 2026

Como a base de **2026 está parcial até abril**, a comparação anual completa pode gerar uma leitura equivocada.  
Por isso, esta análise compara apenas o mesmo recorte temporal nos três anos: **janeiro a abril**.

Essa etapa melhora a consistência metodológica do projeto, pois evita comparar um ano parcial com anos completos.

In [10]:

# Comparação justa entre anos: apenas janeiro a abril
jan_abr = df[df["mes"].isin([1, 2, 3, 4])].copy()

comparativo_jan_abr = (
    jan_abr.groupby("ano", as_index=False)
    .agg(
        acidentes=("id", "count"),
        mortos=("mortos", "sum"),
        feridos=("feridos", "sum"),
        feridos_graves=("feridos_graves", "sum"),
        acidentes_fatais=("acidente_fatal", "sum"),
        gravidade_media=("taxa_gravidade", "mean")
    )
)

comparativo_jan_abr["letalidade_%"] = (
    comparativo_jan_abr["mortos"] / comparativo_jan_abr["acidentes"] * 100
).round(2)

comparativo_jan_abr["fatais_por_1000_acidentes"] = (
    comparativo_jan_abr["acidentes_fatais"] / comparativo_jan_abr["acidentes"] * 1000
).round(2)

# Evolução mensal apenas no recorte Jan-Abr
jan_abr_mensal = (
    jan_abr.groupby(["ano", "mes", "mes_nome"], as_index=False)
    .agg(acidentes=("id", "count"), mortos=("mortos", "sum"))
    .sort_values(["ano", "mes"])
)

jan_abr_mensal["mes_nome"] = pd.Categorical(
    jan_abr_mensal["mes_nome"],
    categories=["Jan", "Fev", "Mar", "Abr"],
    ordered=True
)

fig = px.line(
    jan_abr_mensal,
    x="mes_nome",
    y="acidentes",
    color="ano",
    markers=True,
    title="Comparação Jan–Abr: acidentes por mês nos três anos",
    labels={"mes_nome": "Mês", "acidentes": "Acidentes", "ano": "Ano"}
)

fig.update_layout(template="plotly_white")
fig.show()

display(comparativo_jan_abr)

# Insight automático
linha_maior_volume = comparativo_jan_abr.sort_values("acidentes", ascending=False).iloc[0]
linha_maior_letalidade = comparativo_jan_abr.sort_values("letalidade_%", ascending=False).iloc[0]

fmt_int = lambda v: f"{int(v):,}".replace(",", ".")
fmt_pct = lambda v: f"{float(v):.2f}".replace(".", ",")

texto_insight = f"""
**Insight metodológico — comparação equivalente:**  
Ao restringir a análise ao período de **janeiro a abril**, os três anos passam a ser comparados sob o mesmo intervalo de tempo.  
Nesse recorte, o maior volume de acidentes ocorreu em **{int(linha_maior_volume['ano'])}**, com **{fmt_int(linha_maior_volume['acidentes'])} ocorrências**.  
Já a maior letalidade proporcional ocorreu em **{int(linha_maior_letalidade['ano'])}**, com **{fmt_pct(linha_maior_letalidade['letalidade_%'])}%**.

Essa diferença reforça que a análise deve separar **volume absoluto** de **gravidade proporcional**, principalmente quando existem anos completos e anos parciais na mesma base.
"""

display(Markdown(texto_insight))


,ano,acidentes,mortos,feridos,feridos_graves,acidentes_fatais,gravidade_media,letalidade_%,fatais_por_1000_acidentes
0,2024,22827,1844,26671,6469,1556,1.69,8.08,68.16
1,2025,22561,1787,26351,6121,1537,1.68,7.92,68.13
2,2026,23475,1918,27557,6419,1613,1.69,8.17,68.71



**Insight metodológico — comparação equivalente:**  
Ao restringir a análise ao período de **janeiro a abril**, os três anos passam a ser comparados sob o mesmo intervalo de tempo.  
Nesse recorte, o maior volume de acidentes ocorreu em **2026**, com **23.475 ocorrências**.  
Já a maior letalidade proporcional ocorreu em **2026**, com **8,17%**.

Essa diferença reforça que a análise deve separar **volume absoluto** de **gravidade proporcional**, principalmente quando existem anos completos e anos parciais na mesma base.


## 7. Distribuição geográfica: estados e regiões

A análise por UF mostra a concentração absoluta dos acidentes.  
Porém, para evitar conclusões superficiais, também é importante observar a letalidade proporcional, pois um estado pode ter menos acidentes e, ainda assim, apresentar maior gravidade relativa.


In [11]:

uf = (
    df.groupby("uf", as_index=False)
    .agg(
        acidentes=("id", "count"),
        mortos=("mortos", "sum"),
        feridos=("feridos", "sum"),
        fatais=("acidente_fatal", "sum")
    )
)

uf["letalidade_%"] = (uf["mortos"] / uf["acidentes"] * 100).round(2)

fig = px.bar(
    uf.sort_values("acidentes", ascending=False).head(10).sort_values("acidentes"),
    x="acidentes",
    y="uf",
    orientation="h",
    text="acidentes",
    title="Top 10 UFs por volume de acidentes",
    labels={"uf": "UF", "acidentes": "Acidentes"}
)

fig.update_layout(template="plotly_white")
fig.show()

uf.sort_values("acidentes", ascending=False).head(10)


,uf,acidentes,mortos,feridos,fatais,letalidade_%
10,MG,21851,1828,27574,1498,8.37
23,SC,19414,987,22276,852,5.08
17,PR,17767,1392,19925,1195,7.83
18,RJ,14916,785,17700,716,5.26
22,RS,11593,781,12986,651,6.74
25,SP,10982,504,11893,467,4.59
4,BA,9603,1379,12173,1081,14.36
8,GO,7602,712,8377,578,9.37
15,PE,7329,759,8151,692,10.36
12,MT,6037,561,6425,464,9.29


In [12]:

regiao = (
    df.groupby("regiao", as_index=False)
    .agg(
        acidentes=("id", "count"),
        mortos=("mortos", "sum"),
        feridos=("feridos", "sum"),
        fatais=("acidente_fatal", "sum")
    )
)

regiao["letalidade_%"] = (regiao["mortos"] / regiao["acidentes"] * 100).round(2)

fig = px.bar(
    regiao.sort_values("letalidade_%", ascending=False),
    x="regiao",
    y="letalidade_%",
    text="letalidade_%",
    title="Letalidade por região",
    labels={"regiao": "Região", "letalidade_%": "Letalidade (%)"}
)

fig.update_layout(template="plotly_white")
fig.show()

regiao.sort_values("letalidade_%", ascending=False)


,regiao,acidentes,mortos,feridos,fatais,letalidade_%
2,Norte,9313,1216,10816,1027,13.06
1,Nordeste,37511,4501,43539,3864,12.00
0,Centro-Oeste,20013,1742,21650,1445,8.70
3,Sudeste,53549,3502,64441,3011,6.54
4,Sul,48774,3160,55187,2698,6.48


### Insight 3 — Volume não é o mesmo que risco proporcional

A UF com maior volume absoluto de acidentes foi **MG**, com **21.851 ocorrências**.  
No entanto, quando a análise considera letalidade proporcional entre estados com volume relevante, **MA** apresenta a maior taxa, com **22,54%**.

Por região, a maior letalidade proporcional aparece na região **Norte**, com **13,06%**.  
Esse resultado é relevante porque mostra que decisões baseadas apenas em quantidade podem ocultar áreas com maior severidade. Para prevenção, fiscalização e planejamento, é necessário avaliar simultaneamente volume e gravidade.


### 7.1 Ranking por risco proporcional, não apenas por volume

O ranking por volume mostra onde ocorrem mais acidentes, mas não necessariamente onde os acidentes são mais graves.  
Para aprofundar a análise, foram criados indicadores proporcionais por UF:

- mortos a cada 1.000 acidentes;
- acidentes fatais a cada 1.000 acidentes;
- feridos graves a cada 1.000 acidentes;
- gravidade média por acidente.

Para evitar distorções, o ranking considera apenas UFs com volume mínimo relevante de ocorrências.

In [13]:

# Ranking de risco proporcional por UF
uf_risco = (
    df.groupby("uf", as_index=False)
    .agg(
        acidentes=("id", "count"),
        mortos=("mortos", "sum"),
        feridos=("feridos", "sum"),
        feridos_graves=("feridos_graves", "sum"),
        acidentes_fatais=("acidente_fatal", "sum"),
        total_vitimas=("total_vitimas", "sum"),
        gravidade_media=("taxa_gravidade", "mean")
    )
)

uf_risco["letalidade_%"] = (uf_risco["mortos"] / uf_risco["acidentes"] * 100).round(2)
uf_risco["mortos_por_1000_acidentes"] = (uf_risco["mortos"] / uf_risco["acidentes"] * 1000).round(2)
uf_risco["fatais_por_1000_acidentes"] = (uf_risco["acidentes_fatais"] / uf_risco["acidentes"] * 1000).round(2)
uf_risco["feridos_graves_por_1000_acidentes"] = (uf_risco["feridos_graves"] / uf_risco["acidentes"] * 1000).round(2)
uf_risco["gravidade_media"] = uf_risco["gravidade_media"].round(3)

# mínimo de registros para reduzir distorções por bases pequenas
minimo_acidentes = max(500, int(len(df) * 0.003))
uf_risco_filtrado = uf_risco[uf_risco["acidentes"] >= minimo_acidentes].copy()

ranking_risco = uf_risco_filtrado.sort_values("mortos_por_1000_acidentes", ascending=False).head(10)

fig = px.bar(
    ranking_risco.sort_values("mortos_por_1000_acidentes"),
    x="mortos_por_1000_acidentes",
    y="uf",
    orientation="h",
    text="mortos_por_1000_acidentes",
    title="Ranking de UFs por risco proporcional — mortos a cada 1.000 acidentes",
    labels={"uf": "UF", "mortos_por_1000_acidentes": "Mortos por 1.000 acidentes"}
)

fig.update_layout(template="plotly_white")
fig.show()

display(ranking_risco)

uf_volume_top = uf_risco.sort_values("acidentes", ascending=False).iloc[0]
uf_risco_top = ranking_risco.iloc[0]

fmt_int = lambda v: f"{int(v):,}".replace(",", ".")
fmt_float = lambda v: f"{float(v):.2f}".replace(".", ",")

texto_insight = f"""
**Insight — risco proporcional:**  
A UF com maior volume absoluto foi **{uf_volume_top['uf']}**, com **{fmt_int(uf_volume_top['acidentes'])} acidentes**.  
Porém, no ranking proporcional, a maior taxa entre as UFs com volume relevante aparece em **{uf_risco_top['uf']}**, com **{fmt_float(uf_risco_top['mortos_por_1000_acidentes'])} mortos a cada 1.000 acidentes**.

Isso mostra que o maior volume de registros não representa, necessariamente, o maior risco de morte. Para tomada de decisão, é mais adequado observar volume e severidade de forma combinada.
"""

display(Markdown(texto_insight))


,uf,acidentes,mortos,feridos,feridos_graves,acidentes_fatais,total_vitimas,gravidade_media,letalidade_%,mortos_por_1000_acidentes,fatais_por_1000_acidentes,feridos_graves_por_1000_acidentes
9,MA,2831,638,2934,1100,549,3572,2.10,22.54,225.36,193.92,388.56
13,PA,2419,485,2569,895,423,3054,2.03,20.05,200.50,174.87,369.99
26,TO,1657,270,1910,512,210,2180,1.95,16.29,162.95,126.74,308.99
1,AL,1544,223,1787,590,202,2010,1.97,14.44,144.43,130.83,382.12
4,BA,9603,1379,12173,3289,1081,13552,2.04,14.36,143.60,112.57,342.50
5,CE,3145,401,3586,1104,360,3987,1.87,12.75,127.50,114.47,351.03
16,PI,3472,394,3878,1400,344,4272,1.86,11.35,113.48,99.08,403.23
15,PE,7329,759,8151,2617,692,8910,1.78,10.36,103.56,94.42,357.07
11,MS,3980,376,4275,1159,319,4651,1.65,9.45,94.47,80.15,291.21
8,GO,7602,712,8377,2169,578,9089,1.67,9.37,93.66,76.03,285.32



**Insight — risco proporcional:**  
A UF com maior volume absoluto foi **MG**, com **21.851 acidentes**.  
Porém, no ranking proporcional, a maior taxa entre as UFs com volume relevante aparece em **MA**, com **225,36 mortos a cada 1.000 acidentes**.

Isso mostra que o maior volume de registros não representa, necessariamente, o maior risco de morte. Para tomada de decisão, é mais adequado observar volume e severidade de forma combinada.


### 7.2 Matriz volume x gravidade por UF

A matriz de volume x gravidade combina duas dimensões importantes:

- **volume de acidentes**, representado no eixo X;
- **risco proporcional**, representado no eixo Y;
- **total de vítimas**, representado pelo tamanho da bolha.

Essa visualização ajuda a separar UFs com muitos acidentes daquelas com acidentes proporcionalmente mais graves.

In [14]:

# Matriz volume x gravidade por UF
matriz_uf = uf_risco_filtrado.copy()

limite_volume = matriz_uf["acidentes"].median()
limite_risco = matriz_uf["mortos_por_1000_acidentes"].median()

def classificar_quadrante(linha):
    if linha["acidentes"] >= limite_volume and linha["mortos_por_1000_acidentes"] >= limite_risco:
        return "Alto volume e alto risco"
    elif linha["acidentes"] >= limite_volume and linha["mortos_por_1000_acidentes"] < limite_risco:
        return "Alto volume e menor risco"
    elif linha["acidentes"] < limite_volume and linha["mortos_por_1000_acidentes"] >= limite_risco:
        return "Baixo volume e alto risco"
    else:
        return "Baixo volume e menor risco"

matriz_uf["quadrante"] = matriz_uf.apply(classificar_quadrante, axis=1)

fig = px.scatter(
    matriz_uf,
    x="acidentes",
    y="mortos_por_1000_acidentes",
    size="total_vitimas",
    color="quadrante",
    text="uf",
    hover_name="uf",
    hover_data={
        "acidentes": True,
        "mortos": True,
        "letalidade_%": True,
        "feridos_graves_por_1000_acidentes": True,
        "gravidade_media": True,
        "total_vitimas": True,
        "quadrante": True
    },
    size_max=55,
    title="Matriz volume x gravidade: UFs por acidentes e mortos a cada 1.000 acidentes",
    labels={
        "acidentes": "Volume de acidentes",
        "mortos_por_1000_acidentes": "Mortos por 1.000 acidentes",
        "quadrante": "Classificação"
    }
)

fig.add_vline(x=limite_volume, line_dash="dash", line_color="gray")
fig.add_hline(y=limite_risco, line_dash="dash", line_color="gray")
fig.update_traces(textposition="top center")
fig.update_layout(template="plotly_white")
fig.show()

prioridade = matriz_uf[matriz_uf["quadrante"] == "Alto volume e alto risco"].sort_values(
    ["mortos_por_1000_acidentes", "acidentes"], ascending=False
)

if prioridade.empty:
    prioridade = matriz_uf.sort_values(["mortos_por_1000_acidentes", "acidentes"], ascending=False).head(5)

display(prioridade[[
    "uf", "quadrante", "acidentes", "mortos", "mortos_por_1000_acidentes",
    "feridos_graves_por_1000_acidentes", "gravidade_media", "total_vitimas"
]].head(10))

texto_insight = f"""
**Insight — matriz de priorização:**  
A matriz evidencia que as UFs não devem ser avaliadas apenas pela quantidade total de acidentes.  
O quadrante **alto volume e alto risco** representa as situações mais críticas, pois combina grande número de ocorrências com maior severidade proporcional.

Esse tipo de leitura é mais útil para priorização porque diferencia quatro cenários: alto volume com alto risco, alto volume com menor risco, baixo volume com alto risco e baixo volume com menor risco.
"""

display(Markdown(texto_insight))


,uf,quadrante,acidentes,mortos,mortos_por_1000_acidentes,feridos_graves_por_1000_acidentes,gravidade_media,total_vitimas
4,BA,Alto volume e alto risco,9603,1379,143.60,342.50,2.04,13552
15,PE,Alto volume e alto risco,7329,759,103.56,357.07,1.78,8910
8,GO,Alto volume e alto risco,7602,712,93.66,285.32,1.67,9089
12,MT,Alto volume e alto risco,6037,561,92.93,270.66,1.61,6986



**Insight — matriz de priorização:**  
A matriz evidencia que as UFs não devem ser avaliadas apenas pela quantidade total de acidentes.  
O quadrante **alto volume e alto risco** representa as situações mais críticas, pois combina grande número de ocorrências com maior severidade proporcional.

Esse tipo de leitura é mais útil para priorização porque diferencia quatro cenários: alto volume com alto risco, alto volume com menor risco, baixo volume com alto risco e baixo volume com menor risco.


## 8. Principais causas dos acidentes

A análise das causas ajuda a identificar comportamentos e fatores associados aos acidentes.  
Ela também permite direcionar ações de prevenção, fiscalização e educação no trânsito.


In [15]:

causas = df["causa_acidente"].value_counts().head(10).reset_index()
causas.columns = ["causa_acidente", "acidentes"]

fig = px.bar(
    causas.sort_values("acidentes"),
    x="acidentes",
    y="causa_acidente",
    orientation="h",
    text="acidentes",
    title="Principais causas registradas",
    labels={"causa_acidente": "Causa", "acidentes": "Acidentes"}
)

fig.update_layout(template="plotly_white")
fig.show()

causas


,causa_acidente,acidentes
0,Ausência de reação do condutor,25834
1,Reação tardia ou ineficiente do condutor,25195
2,Acessar a via sem observar a presença dos outr...,16325
3,Condutor deixou de manter distância do veículo...,10260
4,Velocidade Incompatível,9745
5,Manobra de mudança de faixa,9472
6,Ingestão de álcool pelo condutor,8635
7,Demais falhas mecânicas ou elétricas,7858
8,Transitar na contramão,5691
9,Condutor Dormindo,4903


In [16]:

# Causas com maior participação em acidentes fatais
causas_gravidade = (
    df.groupby("causa_acidente", as_index=False)
    .agg(
        acidentes=("id", "count"),
        mortos=("mortos", "sum"),
        acidentes_fatais=("acidente_fatal", "sum")
    )
)

causas_gravidade["letalidade_%"] = (causas_gravidade["mortos"] / causas_gravidade["acidentes"] * 100).round(2)

# Usa mínimo de registros para evitar distorção por causas muito raras
causas_gravidade_relevantes = causas_gravidade[causas_gravidade["acidentes"] >= 300]

causas_gravidade_relevantes.sort_values("letalidade_%", ascending=False).head(10)


,causa_acidente,acidentes,mortos,acidentes_fatais,letalidade_%
60,Suicídio (presumido),424,225,217,53.07
45,Pedestre andava na pista,1460,629,620,43.08
63,Transitar na contramão,5691,2186,1646,38.41
27,Entrada inopinada do pedestre,1523,445,435,29.22
46,Pedestre cruzava a pista fora da faixa,1267,343,332,27.07
66,Ultrapassagem Indevida,4053,922,687,22.75
44,Pedestre - Ingestão de álcool/ substâncias psi...,350,60,59,17.14
20,Deficiência do Sistema de Iluminação/Sinalização,394,63,61,15.99
35,Iluminação deficiente,364,56,52,15.38
38,Mal súbito do condutor,1264,172,148,13.61


### Insight 4 — Predomínio de fatores comportamentais

A principal causa registrada foi **Ausência de reação do condutor**, com **25.834 ocorrências**.  
Além disso, causas relacionadas à reação do condutor, distância insuficiente, velocidade incompatível e ausência de atenção aparecem entre as mais frequentes.

Esse achado sugere que uma parte importante dos acidentes está associada ao comportamento do condutor e à capacidade de reação diante de situações de risco.  
Por isso, os dados reforçam a importância de campanhas educativas, fiscalização, sinalização e medidas preventivas em trechos críticos.


## 9. Fase do dia e severidade

A análise por fase do dia permite verificar se os acidentes se concentram ou se tornam mais graves em determinados períodos, como noite, amanhecer ou pleno dia.


In [17]:

fase = (
    df.groupby("fase_dia", as_index=False)
    .agg(
        acidentes=("id", "count"),
        mortos=("mortos", "sum"),
        fatais=("acidente_fatal", "sum")
    )
)

fase["letalidade_%"] = (fase["mortos"] / fase["acidentes"] * 100).round(2)
fase["acidentes_fatais_%"] = (fase["fatais"] / fase["acidentes"] * 100).round(2)

fig = px.bar(
    fase.sort_values("letalidade_%", ascending=False),
    x="fase_dia",
    y="letalidade_%",
    text="letalidade_%",
    title="Letalidade por fase do dia",
    labels={"fase_dia": "Fase do dia", "letalidade_%": "Letalidade (%)"}
)

fig.update_layout(template="plotly_white")
fig.show()

fase.sort_values("letalidade_%", ascending=False)


,fase_dia,acidentes,mortos,fatais,letalidade_%,acidentes_fatais_%
0,Amanhecer,8016,1080,888,13.47,11.08
2,Plena Noite,57841,6774,5857,11.71,10.13
1,Anoitecer,9099,707,614,7.77,6.75
3,Pleno dia,94204,5560,4686,5.90,4.97


### Insight 5 — Acidentes em períodos de menor visibilidade são mais graves

A fase do dia com maior letalidade foi **Amanhecer**, com **13,47%**.  
Embora o volume absoluto possa ser maior durante o dia, períodos de menor luminosidade tendem a apresentar maior gravidade proporcional.

Esse padrão pode estar relacionado à redução de visibilidade, fadiga, maior velocidade média em vias menos congestionadas e maior dificuldade de reação.  
Assim, a análise mostra que risco não deve ser medido apenas pela frequência, mas também pela severidade.


## 10. Fim de semana x dias úteis

A comparação entre fim de semana e dias úteis ajuda a identificar mudanças no perfil de deslocamento.  
Em finais de semana, pode haver maior presença de viagens de lazer, deslocamentos intermunicipais, direção noturna e maior exposição a comportamentos de risco.


In [18]:

semana = (
    df.groupby("fim_de_semana", as_index=False)
    .agg(
        acidentes=("id", "count"),
        mortos=("mortos", "sum"),
        feridos=("feridos", "sum"),
        fatais=("acidente_fatal", "sum")
    )
)

semana["grupo"] = semana["fim_de_semana"].map({False: "Dias úteis", True: "Fim de semana"})
semana["letalidade_%"] = (semana["mortos"] / semana["acidentes"] * 100).round(2)

fig = px.bar(
    semana,
    x="grupo",
    y="letalidade_%",
    text="letalidade_%",
    title="Letalidade: dias úteis x fim de semana",
    labels={"grupo": "Grupo", "letalidade_%": "Letalidade (%)"}
)

fig.update_layout(template="plotly_white")
fig.show()

semana[["grupo", "acidentes", "mortos", "feridos", "fatais", "letalidade_%"]]


,grupo,acidentes,mortos,feridos,fatais,letalidade_%
0,Dias úteis,115492,8821,132617,7565,7.64
1,Fim de semana,53668,5300,63016,4480,9.88


### Insight 6 — Finais de semana apresentam maior letalidade proporcional

A letalidade nos finais de semana foi de **9,88%**, enquanto nos dias úteis foi de **7,64%**.  
Isso indica que os acidentes de fim de semana, embora não necessariamente representem a maioria absoluta, tendem a ser mais severos proporcionalmente.

Esse comportamento pode estar associado a viagens de lazer, longas distâncias, circulação em horários noturnos e possíveis fatores comportamentais.


## 11. Condição meteorológica e letalidade

A relação entre clima e gravidade dos acidentes é importante porque determinadas condições podem reduzir visibilidade, aderência dos pneus e capacidade de reação do condutor.

Para reduzir distorções, a análise proporcional considera apenas condições meteorológicas com volume mínimo de registros.


In [19]:

clima = (
    df.groupby("condicao_metereologica", as_index=False)
    .agg(
        acidentes=("id", "count"),
        mortos=("mortos", "sum"),
        feridos_graves=("feridos_graves", "sum")
    )
)

clima["letalidade_%"] = (clima["mortos"] / clima["acidentes"] * 100).round(2)

# Para evitar distorções por categorias com poucos registros
clima_filtrado = clima[clima["acidentes"] >= 100].sort_values("letalidade_%", ascending=False)

fig = px.bar(
    clima_filtrado,
    x="condicao_metereologica",
    y="letalidade_%",
    text="letalidade_%",
    title="Letalidade por condição meteorológica",
    labels={"condicao_metereologica": "Condição meteorológica", "letalidade_%": "Letalidade (%)"}
)

fig.update_layout(template="plotly_white")
fig.show()

clima_filtrado


,condicao_metereologica,acidentes,mortos,feridos_graves,letalidade_%
6,Nevoeiro/Neblina,1341,179,406,13.35
4,Ignorado,2208,261,632,11.82
1,Céu Claro,106570,9190,30185,8.62
9,Vento,243,20,73,8.23
7,Nublado,26634,2165,7280,8.13
0,Chuva,16674,1293,4285,7.75
2,Garoa/Chuvisco,6044,401,1344,6.63
8,Sol,9441,612,2575,6.48


### Insight 7 — Nem sempre o clima com maior volume é o mais crítico

Entre as condições meteorológicas com volume relevante de registros, **Nevoeiro/Neblina** apresentou a maior letalidade, com **13,35%**.  
Isso mostra que a análise por taxa é mais adequada do que apenas observar a quantidade total de acidentes por clima.

Condições como neblina, baixa visibilidade ou tempo instável podem não ser as mais frequentes, mas podem aumentar a severidade dos acidentes quando ocorrem.


## 12. Dia da semana x horário

O heatmap permite identificar concentrações temporais, cruzando dia da semana e hora do acidente.  
Essa visualização é útil para apoiar ações de fiscalização em horários críticos e comunicar padrões de forma mais intuitiva.


In [20]:

DIA_ORDEM = ["segunda-feira", "terça-feira", "quarta-feira", "quinta-feira", "sexta-feira", "sábado", "domingo"]

heat = (
    df.groupby(["dia_semana", "hora"], as_index=False)
    .agg(acidentes=("id", "count"))
)

heat["dia_semana"] = pd.Categorical(heat["dia_semana"], categories=DIA_ORDEM, ordered=True)

matriz = (
    heat
    .pivot_table(index="dia_semana", columns="hora", values="acidentes", aggfunc="sum", fill_value=0)
    .reindex(DIA_ORDEM)
)

fig = px.imshow(
    matriz,
    aspect="auto",
    title="Heatmap de acidentes por dia da semana e horário",
    labels=dict(x="Hora", y="Dia da semana", color="Acidentes")
)

fig.update_layout(template="plotly_white")
fig.show()

matriz


hora,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
dia_semana,,,,,,,,,,,,,,,,,,,,,,,,
segunda-feira,579,473,348,381,458,711,1247,1897,1432,1016,1090,1093,1086,1145,1174,1209,1328,1590,1646,1252,856,673,536,441
terça-feira,325,247,228,227,312,556,965,1693,1351,943,989,1002,961,1017,1082,1163,1196,1537,1629,1311,840,708,610,460
quarta-feira,332,334,276,284,340,540,1041,1700,1302,1016,951,975,1029,1007,1063,1086,1199,1593,1747,1408,907,728,642,533
quinta-feira,397,308,280,312,395,601,1023,1690,1296,986,975,973,981,1022,1059,1104,1231,1603,1699,1433,926,815,712,550
sexta-feira,435,358,333,350,454,636,1079,1635,1351,1044,1012,1132,1065,1117,1264,1388,1624,1821,2001,1745,1359,1123,986,763
sábado,626,611,611,683,780,958,1050,1120,1140,1127,1138,1176,1181,1111,1214,1323,1359,1306,1701,1849,1436,1249,1110,941
domingo,768,737,688,765,873,1016,1015,893,870,952,1005,942,929,954,1235,1375,1577,1749,2037,2004,1533,1249,952,750


In [21]:

# Identifica as combinações dia x hora com maior volume
heat_top = heat.sort_values("acidentes", ascending=False).head(10)
heat_top


,dia_semana,hora,acidentes
18,domingo,18,2037
19,domingo,19,2004
114,sexta-feira,18,2001
79,segunda-feira,7,1897
139,sábado,19,1849
113,sexta-feira,17,1821
17,domingo,17,1749
42,quarta-feira,18,1747
115,sexta-feira,19,1745
138,sábado,18,1701


### Interpretação

O heatmap ajuda a identificar janelas de maior concentração de acidentes.  
Horários associados a deslocamento diário, retorno de viagem, tráfego intenso ou baixa luminosidade podem se destacar.

Esse tipo de análise é útil porque permite sair de uma visão geral e direcionar recomendações para momentos específicos da semana, tornando o dashboard mais analítico e menos descritivo.


## 13. Rodovias com maior volume e maior letalidade

A análise por BR permite identificar corredores rodoviários críticos.  
Assim como nos estados, é importante comparar volume absoluto e letalidade proporcional.


In [22]:

br = (
    df.groupby("br", as_index=False)
    .agg(
        acidentes=("id", "count"),
        mortos=("mortos", "sum"),
        feridos=("feridos", "sum")
    )
)

br["letalidade_%"] = (br["mortos"] / br["acidentes"] * 100).round(2)
br["br"] = br["br"].astype(int).astype(str)

fig = px.bar(
    br.sort_values("acidentes", ascending=False).head(10).sort_values("acidentes"),
    x="acidentes",
    y="br",
    orientation="h",
    text="acidentes",
    title="Top 10 BRs por volume de acidentes",
    labels={"br": "BR", "acidentes": "Acidentes"}
)

fig.update_layout(template="plotly_white")
fig.show()

br.sort_values("acidentes", ascending=False).head(10)


,br,acidentes,mortos,feridos,letalidade_%
9,101,30068,1729,34679,5.75
12,116,25875,1777,29283,6.87
63,381,8107,474,9660,5.85
4,40,7959,493,9322,6.19
16,153,6445,633,7334,9.82
20,163,5832,517,6110,8.86
57,364,5396,438,6031,8.12
35,277,4970,368,5713,7.40
62,376,4239,297,4665,7.01
31,262,4085,366,4933,8.96


In [23]:

# Para letalidade, usa-se mínimo de 300 acidentes para reduzir distorções
br_letal = br[br["acidentes"] >= 300].sort_values("letalidade_%", ascending=False).head(10)

fig = px.bar(
    br_letal.sort_values("letalidade_%"),
    x="letalidade_%",
    y="br",
    orientation="h",
    text="letalidade_%",
    title="Top 10 BRs por letalidade proporcional — mínimo de 300 acidentes",
    labels={"br": "BR", "letalidade_%": "Letalidade (%)"}
)

fig.update_layout(template="plotly_white")
fig.show()

br_letal


,br,acidentes,mortos,feridos,letalidade_%
83,423,461,115,564,24.95
24,226,702,167,824,23.79
28,242,750,160,995,21.33
23,222,1327,254,1498,19.14
14,135,1303,245,1417,18.80
29,251,790,148,1149,18.73
1,10,1142,203,1309,17.78
45,316,2911,482,3248,16.56
11,110,777,127,1069,16.34
72,407,688,107,767,15.55


### Interpretação

A BR com maior volume absoluto de acidentes foi a **BR-101**, com **30.068 ocorrências**.  
Por outro lado, entre as rodovias com volume mínimo relevante, a maior letalidade proporcional aparece na **BR-423**, com **24,95%**.

Essa diferença reforça o ponto central da análise: os locais com mais acidentes não são necessariamente os locais com maior risco proporcional.  
Rodovias com maior letalidade podem indicar trechos de pista simples, travessias perigosas, maior velocidade, menor infraestrutura ou condições operacionais mais críticas.


## 14. Matriz de relação: causa x gravidade

Essa análise cruza causa do acidente com gravidade, permitindo identificar quais causas aparecem com maior participação em acidentes fatais, com feridos graves, com feridos leves ou sem vítimas.


In [24]:

top_causas = df["causa_acidente"].value_counts().head(10).index

causa_gravidade = (
    df[df["causa_acidente"].isin(top_causas)]
    .groupby(["causa_acidente", "gravidade"], as_index=False)
    .agg(acidentes=("id", "count"))
)

fig = px.bar(
    causa_gravidade,
    x="acidentes",
    y="causa_acidente",
    color="gravidade",
    orientation="h",
    title="Principais causas por classificação de gravidade",
    labels={"causa_acidente": "Causa", "acidentes": "Acidentes", "gravidade": "Gravidade"}
)

fig.update_layout(template="plotly_white")
fig.show()

causa_gravidade.head(20)


,causa_acidente,gravidade,acidentes
0,Acessar a via sem observar a presença dos outr...,Com feridos graves,4396
1,Acessar a via sem observar a presença dos outr...,Com feridos leves,9607
2,Acessar a via sem observar a presença dos outr...,Fatal,903
3,Acessar a via sem observar a presença dos outr...,Sem vítimas,1419
4,Ausência de reação do condutor,Com feridos graves,5236
5,Ausência de reação do condutor,Com feridos leves,14998
6,Ausência de reação do condutor,Fatal,1681
7,Ausência de reação do condutor,Sem vítimas,3919
8,Condutor Dormindo,Com feridos graves,959
9,Condutor Dormindo,Com feridos leves,2884


In [25]:

# Participação percentual de cada gravidade dentro das principais causas
causa_gravidade_pct = causa_gravidade.copy()
total_por_causa = causa_gravidade_pct.groupby("causa_acidente")["acidentes"].transform("sum")
causa_gravidade_pct["participacao_%"] = (causa_gravidade_pct["acidentes"] / total_por_causa * 100).round(2)

causa_gravidade_pct.sort_values(["causa_acidente", "participacao_%"], ascending=[True, False]).head(30)


,causa_acidente,gravidade,acidentes,participacao_%
1,Acessar a via sem observar a presença dos outr...,Com feridos leves,9607,58.85
0,Acessar a via sem observar a presença dos outr...,Com feridos graves,4396,26.93
3,Acessar a via sem observar a presença dos outr...,Sem vítimas,1419,8.69
2,Acessar a via sem observar a presença dos outr...,Fatal,903,5.53
5,Ausência de reação do condutor,Com feridos leves,14998,58.06
4,Ausência de reação do condutor,Com feridos graves,5236,20.27
7,Ausência de reação do condutor,Sem vítimas,3919,15.17
6,Ausência de reação do condutor,Fatal,1681,6.51
9,Condutor Dormindo,Com feridos leves,2884,58.82
8,Condutor Dormindo,Com feridos graves,959,19.56


### Interpretação

Esse cruzamento ajuda a diferenciar causas frequentes de causas mais severas.  
Uma causa pode aparecer muito em acidentes leves, enquanto outra pode ter menor volume, mas maior peso em acidentes fatais.

Essa distinção é importante para definir prioridades de intervenção, pois ações preventivas não devem considerar apenas as causas mais comuns, mas também aquelas associadas a maior gravidade.


## 15. Síntese dos principais insights

A análise exploratória resultou nos seguintes achados principais:

1. **A base integrada é ampla**, com registros de acidentes entre 2024, 2025 e 2026.
2. **2026 deve ser interpretado como ano parcial**, pois os dados disponíveis vão até abril.
3. **A comparação Jan–Abr torna a leitura anual mais justa**, pois coloca 2024, 2025 e 2026 no mesmo intervalo temporal.
4. **MG lidera em volume absoluto de acidentes**, mas volume não representa necessariamente maior risco proporcional.
5. **O ranking por risco proporcional mostra outra perspectiva**, usando mortos a cada 1.000 acidentes, acidentes fatais a cada 1.000 acidentes e feridos graves a cada 1.000 acidentes.
6. **A matriz volume x gravidade ajuda a priorizar UFs críticas**, separando locais com alto volume e alto risco daqueles com apenas uma dessas características.
7. **A fase do dia influencia a severidade**, com maior atenção para períodos de menor visibilidade.
8. **Finais de semana são proporcionalmente mais letais** do que dias úteis.
9. **Ausência de reação do condutor** aparece entre as causas mais registradas, reforçando a presença de fatores comportamentais nos acidentes.
10. **Condições meteorológicas menos frequentes podem ser mais graves proporcionalmente**, como nevoeiro ou neblina.
11. **A análise por taxa é indispensável**, pois rankings absolutos podem esconder regiões, estados ou rodovias com maior severidade.


## 16. Conclusão

O notebook atende ao pipeline de Ciência de Dados solicitado: aquisição, integração, limpeza, transformação e análise exploratória.  
A inclusão do arquivo de 2024 fortalece a análise, pois permite comparar dois anos completos com o recorte parcial de 2026.

A principal conclusão é que os acidentes em rodovias federais não devem ser avaliados apenas pelo volume absoluto.  
A severidade depende de fatores como fase do dia, região, fim de semana, condição meteorológica, causa do acidente, tipo de rodovia e BR analisada.

Assim, uma abordagem orientada por dados pode apoiar ações mais direcionadas de prevenção, fiscalização e planejamento viário.


## 17. Relação com os dashboards

Os resultados deste notebook servem como base para os dois dashboards do projeto:

- **Dashboard 1 — Visão Executiva:** apresenta KPIs, evolução mensal, ranking de UFs, principais causas, distribuição por gravidade e indicadores consolidados.
- **Dashboard 2 — Exploração Interativa:** aprofunda a análise com filtros por ano, UF, clima, fase do dia e tipo de acidente, além de heatmap, clima x letalidade, comparação regional, causas x gravidade, rodovias críticas e leitura de volume x severidade.

As novas análises acrescentadas fortalecem o storytelling do projeto:

- a comparação **Jan–Abr** evita interpretação incorreta do ano parcial de 2026;
- o ranking por **risco proporcional** mostra que volume absoluto não é suficiente;
- a **matriz volume x gravidade** organiza as UFs por prioridade analítica.

Dessa forma, o notebook documenta o raciocínio analítico e o pipeline de preparação, enquanto o dashboard torna a exploração visual e interativa.  
Essa separação ajuda a atender os critérios de avaliação: qualidade da preparação dos dados, complexidade da análise, qualidade das visualizações, interatividade e clareza na comunicação dos resultados.
